# 04 — Fussdomaene, Sensororientierung, Vorher/Nachher Fussdaten

Ersetzt die frueheren Notebooks 10 und 11 und ergaenzt Teil E.
Benoetigt `fall_model.keras` (01). Rechenzeit: mehrere Stunden (GPU).

| Teil | Frage |
|---|---|
| A | Positionskurve: Basismodell an 9 Koerperpositionen + Fuss (Zero-Shot) |
| B | Lower-Limb-Training A/K/B1/B2 (5-fach, gepaart) |
| C | Orientierung: Erklaert die Achsenlage den Links-rechts-Unterschied? |
| D | Trainingsmatrix W/WK/L0/B2/B2r inkl. Rotationsaugmentierung |
| E | **Vorher/Nachher Fussdaten**: ohne Fussdaten → +ImbalanSense → +Smart-Insole |

Teil E trainiert Smart-Insole mit und dient nur dem Bericht (Lizenz
nicht-kommerziell). Die Pilotaufnahmen in der Sohle werden nicht verwendet.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scipy scikit-learn tensorflow openpyxl

import os, sys, shutil, time, json, pickle, zipfile
from pathlib import Path
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# fall_lib laden: zuerst GitHub, sonst Kopie auf Drive. Ohne die
# aktuelle fall_lib waeren alle Ergebnisse falsch (andere Fensterlaenge).
import urllib.request
_LIB_URL = "https://raw.githubusercontent.com/Alyelb/ML-Modell-for-Fall-detection/main/fall_lib.py"
_LIB_DRIVE = "/content/drive/MyDrive/ML_Fall_Detection/fall_lib.py"
try:
    urllib.request.urlretrieve(_LIB_URL, "/content/fall_lib.py")
    print("[fall_lib] von GitHub geladen")
except Exception as _e:
    print(f"[fall_lib] GitHub nicht erreichbar ({_e}) -- nutze Drive-Kopie")
    shutil.copy(_LIB_DRIVE, "/content/fall_lib.py")
sys.path.insert(0, "/content")
import importlib, fall_lib as fl
importlib.reload(fl)
for _f in ("new_run", "build_windows", "znorm", "load_cappimu", "load_pilot",
           "augment_orientation", "rotation_between", "voting_eval", "finetune"):
    assert hasattr(fl, _f), f"fall_lib veraltet ({_f} fehlt) -- neue Version auf Drive legen."
assert fl.WIN_LEN == 100, "falsche Fensterlaenge -- fall_lib pruefen"
print("[fall_lib] Fensterlaenge", fl.WIN_LEN, "bei", fl.FS_TARGET, "Hz")

BASE  = "/content/drive/MyDrive/ML_Fall_Detection"
fl.RESULTS_ROOT = f"{BASE}/ergebnisse"
MODEL = f"{BASE}/ergebnisse/fall_model.keras"        # Basismodell (Teil 01)
PILOT = f"{BASE}/Pilotdaten"
CACHE = Path(f"{BASE}/cache"); CACHE.mkdir(exist_ok=True)


def hole(name, ziel):
    """Kopiert einen Datensatz von Drive auf die lokale Platte
    (Drive ist fuer tausende Kleindateien zu langsam). Gibt None
    zurueck, wenn der Ordner fehlt."""
    src = f"{BASE}/{name}"
    if Path(ziel).exists():
        print(f"[ok] {name} bereits lokal"); return ziel
    if not Path(src).exists():
        print(f"[!] {src} nicht gefunden. Vorhanden:")
        for p in sorted(Path(BASE).iterdir()):
            print("   ", p.name)
        return None
    t = time.time(); shutil.copytree(src, ziel)
    print(f"[ok] {name} kopiert in {time.time()-t:.0f}s")
    return ziel


def cached(name, fn):
    """Geladene Trials auf Drive zwischenspeichern (Laden dauert lange)."""
    p = CACHE / f"{name}.pkl"
    if p.exists():
        print(f"[cache] {name}")
        return pickle.load(open(p, "rb"))
    r = fn(); pickle.dump(r, open(p, "wb")); return r


from tensorflow import keras
def lade_modell(pfad):
    if not Path(pfad).exists():
        raise FileNotFoundError(f"{pfad} fehlt -- vorheriges Notebook ausfuehren.")
    m = keras.models.load_model(pfad, compile=False)
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return m

print("Konfiguration:", fl.default_config())

In [ ]:
basis = lade_modell(MODEL)
run = fl.new_run("10_fussdomaene")
ergebnisse = {}

# Teil A/B — Positionskurve und Lower-Limb-Modell

## Datenpfade

CAPPIMU wird **direkt von Drive** gelesen: wenige große Dateien
(~30 MB je Position und Proband), das Kopieren aller ~20 GB lohnt
nicht. Gelesen werden nur die benötigten Positionen.

In [ ]:
# CAPPIMU: Ordner, der subject_1 ... subject_30 enthaelt
CAP_ROOT = next(Path(f"{BASE}/CAPPIMU Dataset").rglob("subject_1")).parent
print("CAPPIMU:", CAP_ROOT)

# ImbalanSense: entpackte .txt oder raw_recordings.zip
IMB = Path("/content/ImbalanSense")
if not IMB.exists():
    src = Path(f"{BASE}/ImbalanSense")
    if any(src.rglob("*.txt")):
        hole("ImbalanSense", str(IMB))
    else:
        z = next(src.rglob("raw_recordings*.zip"))
        zipfile.ZipFile(z).extractall(IMB)
print("ImbalanSense:", sum(1 for _ in IMB.rglob("*.txt")), "Dateien")

# Smart-Insole: nur Annotated_Data
SI = Path("/content/SmartInsole")
if not SI.exists():
    src = next(Path(f"{BASE}/SmartInsole_Dataset_v1_0_Public").rglob("Annotated_Data"))
    shutil.copytree(src, SI)
print("Smart-Insole:", sum(1 for _ in SI.rglob("*.xlsx")), "Dateien")

## Teil A — Positionskurve (Zero-Shot)

Kennzahlen je Position:
* **Se** auf aufprallzentrierten Sturzfenstern
* **FP/Std** auf allen ADL-Fenstern (Schrittweite 1 s, Schwelle 0,5)
* **Peak Sturz** (Median) und **Peak ADL** (99. Perzentil) von |a| in g —
  je kleiner der Abstand, desto schwerer die Trennung

In [ ]:
def fenster(trials):
    X, y, g, a = fl.build_windows(trials, mode="standard")
    peaks = fl.acc_magnitude(X).max(axis=1)        # vor Normalisierung
    return fl.znorm(X), y, g, a, peaks

def fp_pro_stunde(n_fp, n_adl):
    h = n_adl * fl.WIN_SEC * (1 - fl.OVERLAP) / 3600
    return round(n_fp / h, 2) if h else None

def bewerte(model, X, y, peaks, name, thr=0.5):
    r = fl.evaluate(model, X, y, name, threshold=thr)
    r["fp_per_hour"] = fp_pro_stunde(r["FP"], int((y == 0).sum()))
    r["n_falls"] = int(y.sum()); r["n_adl"] = int((y == 0).sum())
    r["peak_fall_median_g"] = round(float(np.median(peaks[y == 1])), 2) if y.sum() else None
    r["peak_adl_p99_g"] = round(float(np.percentile(peaks[y == 0], 99)), 2)
    return r

kurve = {}
for pos in fl.CAPPIMU_POSITIONS:
    t = time.time()
    tr = cached(f"cappimu_{pos}", lambda: fl.load_cappimu(CAP_ROOT, pos))
    if not tr:
        print(f"[!] {pos}: keine Daten"); continue
    X, y, g, a, pk = fenster(tr)
    kurve[f"CAPPIMU {pos}"] = bewerte(basis, X, y, pk, f"CAPPIMU {pos}")
    print(f"     FP/Std={kurve[f'CAPPIMU {pos}']['fp_per_hour']}  ({time.time()-t:.0f}s)")
    del tr, X

In [ ]:
imb = cached("imbalansense", lambda: fl.load_imbalansense(IMB))
si  = cached("smart_insole", lambda: fl.load_smart_insole(SI))

X_imb, y_imb, g_imb, a_imb, pk_imb = fenster(imb)
X_si,  y_si,  g_si,  a_si,  pk_si  = fenster(si)
grp_si = np.array([s.split("_")[1].rstrip("0123456789") for s in g_si])

kurve["Fuss ImbalanSense"] = bewerte(basis, X_imb, y_imb, pk_imb, "Fuss ImbalanSense")
kurve["Fuss Smart-Insole"] = bewerte(basis, X_si, y_si, pk_si, "Fuss Smart-Insole")

print("\nImbalanSense nach Bedingung:")
for c in ["norm", "vr", "shaking"]:
    m = np.array([s.startswith(c) for s in a_imb])
    if m.any(): kurve[f"Fuss IMB {c}"] = bewerte(basis, X_imb[m], y_imb[m], pk_imb[m], f"  {c}")
print("\nSmart-Insole nach Gruppe:")
for gr in ["S", "EL", "PD"]:
    m = grp_si == gr
    if m.any(): kurve[f"Fuss SI {gr}"] = bewerte(basis, X_si[m], y_si[m], pk_si[m], f"  {gr}")
ergebnisse["positionskurve"] = kurve

In [ ]:
import matplotlib.pyplot as plt

reihen = [p for p in ["head", "chest", "arm_l", "arm_r", "wrist_l", "wrist_r",
                      "pocket", "knee_l", "knee_r"] if f"CAPPIMU {p}" in kurve]
namen  = [f"CAPPIMU {p}" for p in reihen] + ["Fuss ImbalanSense", "Fuss Smart-Insole"]
labels = reihen + ["Fuß (IMB)", "Fuß (SI)"]

print(f"{'Position':<22}{'Se':>7}{'Sp':>7}{'FP/Std':>9}{'Peak Sturz':>12}{'Peak ADL p99':>14}")
print("-" * 71)
for n in namen:
    r = kurve[n]
    se = f"{r['Se']:.3f}" if r["n_falls"] else "  -  "
    pf = f"{r['peak_fall_median_g']:.2f}" if r["peak_fall_median_g"] else "-"
    print(f"{n:<22}{se:>7}{r['Sp']:>7.3f}{r['fp_per_hour']:>9.1f}{pf:>12}{r['peak_adl_p99_g']:>14.2f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
x = np.arange(len(namen))
se = [kurve[n]["Se"] if kurve[n]["n_falls"] else np.nan for n in namen]
fp = [kurve[n]["fp_per_hour"] for n in namen]
ax[0].plot(x, se, "o-", color="#80BA24", label="Se")
ax0b = ax[0].twinx(); ax0b.bar(x, fp, alpha=.35, color="#9C132E", label="FP/Std")
ax[0].set_xticks(x); ax[0].set_xticklabels(labels, rotation=35, ha="right")
ax[0].set_ylabel("Sensitivität"); ax0b.set_ylabel("Fehlalarme / Stunde")
ax[0].set_title("Basismodell (SisFall) je Position")

pf = [kurve[n]["peak_fall_median_g"] or np.nan for n in namen]
pa = [kurve[n]["peak_adl_p99_g"] for n in namen]
ax[1].plot(x, pf, "o-", color="#80BA24", label="Sturz (Median)")
ax[1].plot(x, pa, "s--", color="#4A5C66", label="ADL (99. Perzentil)")
ax[1].set_xticks(x); ax[1].set_xticklabels(labels, rotation=35, ha="right")
ax[1].set_ylabel("max |a| je Fenster [g]"); ax[1].legend()
ax[1].set_title("Abstand Sturz – Alltag")
plt.tight_layout()
plt.savefig(run / "positionskurve.png", dpi=200, bbox_inches="tight")
plt.show()

## Teil B — Lower-Limb-Training

Fünffache, subjektunabhängige Kreuzvalidierung, getrennt für CAPPIMU
(Probanden) und ImbalanSense (Probanden). Falte *i* testet jeweils auf
den Probanden beider Datensätze, die nicht im Training waren.
Smart-Insole ist in keiner Falte im Training.

ADL-Fenster werden **für das Training** gekappt (Rechenzeit); bewertet
wird immer auf allen Testfenstern.

In [ ]:
N_FOLDS   = 5
MAX_ADL_CAP = 20000      # ADL-Trainingsfenster je Falte aus CAPPIMU
MAX_ADL_IMB = 10000      # ADL-Trainingsfenster je Falte aus ImbalanSense
rng = np.random.default_rng(42)

def kappe(idx, y, max_adl):
    f = idx[y[idx] == 1]; a = idx[y[idx] == 0]
    if len(a) > max_adl:
        a = rng.choice(a, max_adl, replace=False)
    return np.r_[f, a]

# CAPPIMU Knie (beide Beine, gleicher Proband -> gleiche Gruppe)
knie = cached("cappimu_knee_l", lambda: fl.load_cappimu(CAP_ROOT, "knee_l")) + \
       cached("cappimu_knee_r", lambda: fl.load_cappimu(CAP_ROOT, "knee_r"))
X_c, y_c, g_c, a_c, pk_c = fenster(knie)
print(f"CAPPIMU Knie: {len(y_c)} Fenster, {int(y_c.sum())} Stuerze, {len(np.unique(g_c))} Probanden")
print(f"ImbalanSense: {len(y_imb)} Fenster, {len(np.unique(g_imb))} Probanden")
print(f"Smart-Insole: {len(y_si)} Fenster (nur Test)")

# KFall-Zwischenstufe (wie Notebook 07)
kf = fl.load_kfall(hole("KFALL/sensor_data_new", "/content/KFall"),
                   label_dir=hole("KFALL/label_data_new", "/content/KFall_labels"))
X_kf, y_kf, _, _ = fl.build_windows(kf, mode="standard")
X_kf = fl.znorm(X_kf)
t = time.time(); m_kfall = fl.finetune(basis, X_kf, y_kf)
print(f"KFall-Fine-Tuning: {time.time()-t:.0f}s")
del X_kf

In [ ]:
sp_cap = fl.group_kfold_splits(g_c, y_c, n_splits=N_FOLDS)
sp_imb = fl.group_kfold_splits(g_imb, None, n_splits=N_FOLDS)

def alle_tests(m, te_c, te_i, tag):
    return {
        "knie": fl.evaluate(m, X_c[te_c], y_c[te_c], f"{tag} Knie"),
        "imb":  fl.evaluate(m, X_imb[te_i], y_imb[te_i], f"{tag} Fuss IMB"),
        "si":   fl.evaluate(m, X_si, y_si, f"{tag} Fuss SI"),
    }

falten = {k: [] for k in ["A", "K", "B1", "B2"]}
for i, ((tr_c, te_c), (tr_i, te_i)) in enumerate(zip(sp_cap, sp_imb), 1):
    print(f"\nFalte {i}/{N_FOLDS}: Knie-Test {len(te_c)} Fenster "
          f"({int(y_c[te_c].sum())} Stuerze), IMB-Test {len(te_i)} Fenster")
    t = time.time()
    falten["A"].append(alle_tests(basis,   te_c, te_i, "  A "))
    falten["K"].append(alle_tests(m_kfall, te_c, te_i, "  K "))

    i_c = kappe(tr_c, y_c, MAX_ADL_CAP)
    m_b1 = fl.finetune(m_kfall, X_c[i_c], y_c[i_c])
    falten["B1"].append(alle_tests(m_b1, te_c, te_i, "  B1"))

    i_i = rng.choice(tr_i, min(MAX_ADL_IMB, len(tr_i)), replace=False)
    X_tr = np.concatenate([X_c[i_c], X_imb[i_i]])
    y_tr = np.concatenate([y_c[i_c], y_imb[i_i]])
    m_b2 = fl.finetune(m_kfall, X_tr, y_tr)
    falten["B2"].append(alle_tests(m_b2, te_c, te_i, "  B2"))
    print(f"  ({time.time()-t:.0f}s)")
    del m_b1, m_b2

## Zusammengefasstes Ergebnis

In [ ]:
h = lambda n: n * fl.WIN_SEC * (1 - fl.OVERLAP) / 3600
zusammen = {}
print(f"{'Var.':<5}{'Knie Se':>9}{'95%-KI':>18}{'Knie Sp':>9}{'IMB FP/Std':>12}{'SI FP/Std':>11}")
print("-" * 64)
for k, fl_ in falten.items():
    knie_p = fl.pool_folds([f["knie"] for f in fl_])
    imb_fp = sum(f["imb"]["FP"] for f in fl_); imb_n = sum(f["imb"]["FP"] + f["imb"]["TN"] for f in fl_)
    si_fp  = np.mean([f["si"]["FP"] for f in fl_]); si_n = len(y_si)
    zusammen[k] = {"knie": knie_p,
                   "imb_fp_per_hour": round(imb_fp / h(imb_n), 2),
                   "si_fp_per_hour": round(si_fp / h(si_n), 2)}
    ci = knie_p["Se_ci95"]
    print(f"{k:<5}{knie_p['Se']:>9.3f}   [{ci[0]:.3f}; {ci[1]:.3f}]{knie_p['Sp']:>9.3f}"
          f"{zusammen[k]['imb_fp_per_hour']:>12.1f}{zusammen[k]['si_fp_per_hour']:>11.1f}")
ergebnisse["lower_limb"] = {"zusammen": zusammen, "falten": falten}

## Gepaarte Vergleiche

* **B1 vs K:** Bringt Knie-Training etwas gegenüber dem Rumpfmodell?
* **B2 vs B1:** Helfen zusätzliche Fußdaten (ImbalanSense) — oder schaden
  sie der Sensitivität wie das Hard-Negative-Training in Notebook 08?

In [ ]:
vgl = {}
for a, b, met, quelle in [("K", "B1", "Se", "knie"), ("K", "B1", "Sp", "knie"),
                          ("B1", "B2", "Se", "knie"), ("B1", "B2", "Sp", "knie"),
                          ("B1", "B2", "Sp", "imb"),  ("B1", "B2", "Sp", "si")]:
    print(f"\n{b} vs {a} — {quelle}")
    vgl[f"{b}_vs_{a}_{met}_{quelle}"] = fl.paired_comparison(
        [f[quelle] for f in falten[a]], [f[quelle] for f in falten[b]], a, b, metric=met)
ergebnisse["vergleiche"] = vgl

## Endmodell für den Pilotversuch

B2 auf **allen** CAPPIMU-Knie- und ImbalanSense-Daten trainiert.
Wird in Notebook 09 als zweites Modell neben dem Basismodell auf die
eigenen ESP32-Aufnahmen angewendet.

In [ ]:
i_c = kappe(np.arange(len(y_c)), y_c, MAX_ADL_CAP)
i_i = rng.choice(np.arange(len(y_imb)), min(MAX_ADL_IMB, len(y_imb)), replace=False)
m_final = fl.finetune(m_kfall, np.concatenate([X_c[i_c], X_imb[i_i]]),
                      np.concatenate([y_c[i_c], y_imb[i_i]]))
pfad = f"{BASE}/ergebnisse/fall_model_lowerlimb.keras"
m_final.save(pfad)
print("gesichert:", pfad)

fl.save_results(run, ergebnisse,
    config={**fl.default_config(), "n_folds": N_FOLDS,
            "max_adl_cap": MAX_ADL_CAP, "max_adl_imb": MAX_ADL_IMB,
            "cappimu_ref": "chest", "cappimu_min_g": 2.5},
    notes=("Positionskurve (CAPPIMU 9 Positionen + Fuss) und "
           "Lower-Limb-Training K/B1/B2 mit 5-facher CV. "
           "Smart-Insole nur externer Test."))

In [ ]:
# Speicher freigeben, bevor Teil C die Rohdaten neu laedt
for _v in ["X_c", "y_c", "g_c", "a_c", "pk_c", "X_imb", "X_si", "knie", "imb", "si", "m_final"]:
    globals().pop(_v, None)
import gc; gc.collect()

# Teil C/D — Orientierung und Trainingsmatrix

In [ ]:
run = fl.new_run("11_orientierung_matrix")
ergebnisse = {}

## Daten

Rohfenster (g, °/s) werden unnormiert gehalten, weil Drehung und
Spiegelung **vor** der Z-Normalisierung erfolgen müssen. Normiert wird
erst unmittelbar vor der Vorhersage bzw. dem Training.

In [ ]:
CAP_ROOT = next(Path(f"{BASE}/CAPPIMU Dataset").rglob("subject_1")).parent

def roh(trials):
    X, y, g, a = fl.build_windows(trials, mode="standard")
    return X, y, g, a

def vorhersage(m, X, bs=2048):
    out = []
    for i in range(0, len(X), bs):
        out.append(m.predict(fl.znorm(X[i:i+bs]), batch_size=512, verbose=0).ravel())
    return np.concatenate(out) if out else np.empty(0)

def kennzahlen(p, y, thr=0.5):
    tp = int(((p > thr) & (y == 1)).sum()); fn = int(((p <= thr) & (y == 1)).sum())
    fp = int(((p > thr) & (y == 0)).sum()); tn = int(((p <= thr) & (y == 0)).sum())
    h = (fp + tn) * fl.WIN_SEC * (1 - fl.OVERLAP) / 3600
    return {"Se": round(tp / (tp + fn), 4) if tp + fn else None,
            "Sp": round(tn / (tn + fp), 4) if tn + fp else None,
            "fp_per_hour": round(fp / h, 1) if h else None,
            "TP": tp, "FN": fn, "FP": fp, "TN": tn}

# Pilotdaten (eigene Hardware)
pilot = [t for t in fl.load_pilot(PILOT) if "INSOLE" not in t["position"]]   # Sohle ausgeschlossen
PIL = {}
for pos in sorted({t["position"] for t in pilot}):
    PIL[pos] = roh([t for t in pilot if t["position"] == pos])

## Teil A — Orientierung (Basismodell)

Für jedes Paar linker/rechter Sensoren wird die linke Seite an jeder
der drei Achsenebenen gespiegelt; zusätzlich 20 zufällige Drehungen.
Die Spannweite über die Drehungen zeigt, wie stark das Ergebnis allein
von der Einbaulage abhängt. Die Betragsmerkmale des Random Forest
dienen als Kontrolle: Sie müssen unverändert bleiben.

In [ ]:
N_ROT = 20
ROT = fl.random_rotation_matrices(N_ROT, seed=7)
I3 = np.eye(3)

def orientierung(m, X, y, name):
    r = {"original": kennzahlen(vorhersage(m, X), y)}
    for k in range(3):
        r[f"spiegel_{'xyz'[k]}"] = kennzahlen(vorhersage(m, fl.rotate_windows(X, I3, k)), y)
    se = [kennzahlen(vorhersage(m, fl.rotate_windows(X, R)), y)["Se"] for R in ROT]
    r["rot_Se_mean"] = float(np.mean(se)); r["rot_Se_min"] = float(np.min(se))
    r["rot_Se_max"] = float(np.max(se))
    sp = " ".join(f"{k[-1]}:{r[k]['Se']:.2f}" for k in r if k.startswith("spiegel"))
    print(f"  {name:<22} Se orig {r['original']['Se']:.3f} | gespiegelt {sp} | "
          f"Drehung {r['rot_Se_mean']:.2f} [{r['rot_Se_min']:.2f}; {r['rot_Se_max']:.2f}]")
    return r

teilA = {}
for paar in [("arm_l", "arm_r"), ("wrist_l", "wrist_r"), ("knee_l", "knee_r")]:
    for pos in paar:
        tr = cached(f"cappimu_{pos}", lambda: fl.load_cappimu(CAP_ROOT, pos))
        if not tr:
            continue
        X, y, g, a = roh(tr)
        f = y == 1                                   # Orientierung betrifft v. a. Se
        teilA[f"CAPPIMU {pos}"] = orientierung(basis, X[f], y[f], f"CAPPIMU {pos}")
        del X
for pos, (X, y, g, a) in PIL.items():
    if y.sum():
        teilA[f"Pilot {pos}"] = orientierung(basis, X[y == 1], y[y == 1], f"Pilot {pos}")

# Kontrolle Random-Forest-Merkmale
X, y, _, _ = PIL[next(iter(PIL))]
d = np.abs(fl.extract_features(X[:200]) - fl.extract_features(fl.rotate_windows(X[:200], ROT[0], 0)))
print(f"\nBetragsmerkmale nach Drehung+Spiegelung: max. rel. Abweichung "
      f"{(d.max(0) / (np.abs(fl.extract_features(X[:200])).max(0) + 1e-9)).max():.1e}")
ergebnisse["teilA_orientierung"] = teilA

**Lesart:** Erreicht die gespiegelte linke Seite das Niveau der rechten,
erklärt die Orientierung die Seitenlücke. Liegt die Spannweite über
zufällige Drehungen weit auseinander, ist das CNN für eine Sohle mit
wechselnder Einbaulage ohne Gegenmaßnahme nicht geeignet.

## Teil B — Trainingsmatrix

Fünffache, subjektunabhängige Kreuzvalidierung über CAPPIMU-Schienbein
und ImbalanSense wie in Notebook 10. In jeder Falte werden L0, B2 und
B2r neu trainiert; W und WK sind faltenunabhängig.

B2 und B2r werden hier **vollständig** nachtrainiert (), anders
als in Notebook 10 (erste Faltungsschicht eingefroren). Eine
Drehinvarianz muss bereits in der ersten Schicht entstehen; mit
eingefrorener Schicht wäre der Vergleich zugunsten von B2 verzerrt.
Der Unterschied zwischen B2 und B2r ist damit allein die
Augmentierung.

In [ ]:
N_FOLDS = 5
MAX_ADL_CAP, MAX_ADL_IMB = 20000, 10000
rng = np.random.default_rng(42)

knie = cached("cappimu_knee_l", lambda: fl.load_cappimu(CAP_ROOT, "knee_l")) + \
       cached("cappimu_knee_r", lambda: fl.load_cappimu(CAP_ROOT, "knee_r"))
Xc, yc, gc, ac = roh(knie); del knie
def aus_cache(name):
    p = CACHE / f"{name}.pkl"
    if not p.exists():
        raise FileNotFoundError(f"{p} fehlt -- bitte zuerst Notebook 10 ausfuehren.")
    return pickle.load(open(p, "rb"))

imb = aus_cache("imbalansense")
Xi, yi, gi, ai = roh(imb); del imb
si = aus_cache("smart_insole")
Xs, ys, gs, as_ = roh(si); del si
print(f"Schienbein {len(yc)} Fenster ({int(yc.sum())} Stuerze), "
      f"ImbalanSense {len(yi)}, Smart-Insole {len(ys)}")

# WK: einmal berechnen und sichern
WK_PFAD = f"{BASE}/ergebnisse/fall_model_kfall.keras"
if Path(WK_PFAD).exists():
    m_wk = lade_modell(WK_PFAD)
else:
    kf = fl.load_kfall(hole("KFALL/sensor_data_new", "/content/KFall"),
                       label_dir=hole("KFALL/label_data_new", "/content/KFall_labels"))
    Xk, yk, _, _ = fl.build_windows(kf, mode="standard"); del kf
    m_wk = fl.finetune(basis, fl.znorm(Xk), yk); del Xk
    m_wk.save(WK_PFAD)

def kappe(idx, y, n):
    f = idx[y[idx] == 1]; a = idx[y[idx] == 0]
    return np.r_[f, rng.choice(a, n, replace=False) if len(a) > n else a]

def trainingssatz(tr_c, tr_i, aug=False):
    i_c = kappe(tr_c, yc, MAX_ADL_CAP)
    i_i = rng.choice(tr_i, min(MAX_ADL_IMB, len(tr_i)), replace=False)
    X = np.concatenate([Xc[i_c], Xi[i_i]]); y = np.concatenate([yc[i_c], yi[i_i]])
    if aug:
        X, y = fl.augment_orientation(X, y, copies=1, seed=int(rng.integers(1e6)))
    return fl.znorm(X), y

def von_null(X, y):
    m = fl.build_cnn()
    return fl.train(m, X, y)

In [ ]:
sp_c = fl.group_kfold_splits(gc, yc, n_splits=N_FOLDS)
sp_i = fl.group_kfold_splits(gi, None, n_splits=N_FOLDS)
KETTEN = ["W", "WK", "L0", "B2", "B2r"]
falten = {k: [] for k in KETTEN}

def bewerte(m, te_c, te_i, k_rot):
    r = {"knie": kennzahlen(vorhersage(m, Xc[te_c]), yc[te_c]),
         "imb":  kennzahlen(vorhersage(m, Xi[te_i]), yi[te_i]),
         "si":   kennzahlen(vorhersage(m, Xs), ys)}
    f = te_c[yc[te_c] == 1]
    se = [kennzahlen(vorhersage(m, fl.rotate_windows(Xc[f], ROT[j])), yc[f])["Se"]
          for j in k_rot]
    r["knie_rot_Se"] = float(np.mean(se))
    for pos, (X, y, _, _) in PIL.items():
        r[f"pilot_{pos}"] = kennzahlen(vorhersage(m, X), y)
    return r

for i, ((tr_c, te_c), (tr_i, te_i)) in enumerate(zip(sp_c, sp_i), 1):
    t = time.time()
    k_rot = range(5)                                  # 5 Drehungen je Falte
    print(f"\nFalte {i}/{N_FOLDS}")
    falten["W"].append(bewerte(basis, te_c, te_i, k_rot))
    falten["WK"].append(bewerte(m_wk, te_c, te_i, k_rot))
    X, y = trainingssatz(tr_c, tr_i)
    m = von_null(X, y);           falten["L0"].append(bewerte(m, te_c, te_i, k_rot)); del m
    m = fl.finetune(m_wk, X, y, freeze=0);  falten["B2"].append(bewerte(m, te_c, te_i, k_rot)); del m
    X, y = trainingssatz(tr_c, tr_i, aug=True)
    m = fl.finetune(m_wk, X, y, freeze=0);  falten["B2r"].append(bewerte(m, te_c, te_i, k_rot)); del m, X
    for k in KETTEN:
        r = falten[k][-1]
        print(f"  {k:<4} Knie Se {r['knie']['Se']:.3f} (gedreht {r['knie_rot_Se']:.3f})  "
              f"IMB FP/h {r['imb']['fp_per_hour']:>6}  SI FP/h {r['si']['fp_per_hour']:>6}")
    print(f"  ({time.time()-t:.0f}s)")

## Ergebnis

In [ ]:
def pool(liste, schl):
    tp = sum(r[schl]["TP"] for r in liste); fn = sum(r[schl]["FN"] for r in liste)
    fp = sum(r[schl]["FP"] for r in liste); tn = sum(r[schl]["TN"] for r in liste)
    h = (fp + tn) * fl.WIN_SEC * (1 - fl.OVERLAP) / 3600
    return {"Se": tp / (tp + fn) if tp + fn else None,
            "Se_ci95": fl.wilson_ci(tp, tp + fn) if tp + fn else None,
            "fp_per_hour": fp / h if h else None}

pos_pilot = list(PIL)
kopf = f"{'Kette':<5}{'Knie Se':>9}{'gedreht':>9}{'IMB FP/h':>10}{'SI FP/h':>9}"
for p in pos_pilot:
    kopf += f"{p[:10]+' Se':>14}{'FP/h':>7}"
print(kopf); print("-" * len(kopf))
tabelle = {}
for k in KETTEN:
    L = falten[k]
    z = {"knie": pool(L, "knie"), "knie_rot_Se": float(np.mean([r["knie_rot_Se"] for r in L])),
         "imb": pool(L, "imb"), "si": pool(L, "si")}
    zeile = (f"{k:<5}{z['knie']['Se']:>9.3f}{z['knie_rot_Se']:>9.3f}"
             f"{z['imb']['fp_per_hour']:>10.1f}{z['si']['fp_per_hour']:>9.1f}")
    for p in pos_pilot:
        z[p] = pool(L, f"pilot_{p}")
        se = z[p]["Se"]
        zeile += f"{(f'{se:.2f}' if se is not None else '-'):>14}{z[p]['fp_per_hour']:>7.0f}"
    print(zeile); tabelle[k] = z
ergebnisse["teilB_matrix"] = {"tabelle": tabelle, "falten": falten}

## Gepaarte Vergleiche

* **B2 vs L0** — Lohnt das Vortraining auf Rumpfdaten?
* **B2r vs B2** — Kostet die Augmentierung Sensitivität, und gewinnt sie
  Robustheit gegenüber Drehung?

In [ ]:
vgl = {}
for a, b, schl, met in [("L0", "B2", "knie", "Se"), ("L0", "B2", "imb", "Sp"),
                        ("B2", "B2r", "knie", "Se"), ("B2", "B2r", "knie_rot_Se", "Se"),
                        ("B2", "B2r", "imb", "Sp"), ("B2", "B2r", "si", "Sp")]:
    print(f"\n{b} vs {a} — {schl}")
    A = [{"Se": r[schl]} if schl == "knie_rot_Se" else r[schl] for r in falten[a]]
    B = [{"Se": r[schl]} if schl == "knie_rot_Se" else r[schl] for r in falten[b]]
    vgl[f"{b}_vs_{a}_{schl}_{met}"] = fl.paired_comparison(A, B, a, b, metric=met)
ergebnisse["vergleiche"] = vgl

## Endmodell mit Augmentierung

In [ ]:
alle_c = np.arange(len(yc)); alle_i = np.arange(len(yi))
X, y = trainingssatz(alle_c, alle_i, aug=True)
m_final = fl.finetune(m_wk, X, y, freeze=0); del X
pfad = f"{BASE}/ergebnisse/fall_model_lowerlimb_aug.keras"
m_final.save(pfad); print("gesichert:", pfad)

fl.save_results(run, ergebnisse,
    config={**fl.default_config(), "n_folds": N_FOLDS, "n_rot_teilA": N_ROT,
            "max_adl_cap": MAX_ADL_CAP, "max_adl_imb": MAX_ADL_IMB,
            "augmentierung": "1 Kopie, zufaellige Drehung, 50 % Spiegelung"},
    notes="Teil A Orientierung (Spiegelung/Drehung), Teil B Trainingsmatrix W/WK/L0/B2/B2r.")

# Teil E — Vorher/Nachher: Fussdaten im Training

Gleiche Falten wie Teil D (CAPPIMU-Schienbein, ImbalanSense), zusaetzlich
fuenf Falten ueber die Smart-Insole-Probanden. Alle drei Varianten starten
von WK, werden vollstaendig nachtrainiert und orientierungsaugmentiert;
sie unterscheiden sich nur in den Fussdaten:

| Variante | Fussdaten im Training |
|---|---|
| ohne Fussdaten (B1r) | keine |
| + ImbalanSense (B2r) | ImbalanSense (Moticon, CC BY 4.0) |
| + ImbalanSense + Smart-Insole | zusaetzlich Smart-Insole der Trainingsprobanden |

Bewertet wird Smart-Insole immer auf Probanden, die nicht im Training waren.

In [ ]:
# ---------------------------------------------------------------
# Teil E: Vorher/Nachher -- Smart-Insole im Training
# ---------------------------------------------------------------
run_si = fl.new_run("12_vorher_nachher_fussdaten")
MAX_ADL_SI = 10000
sp_s = fl.group_kfold_splits(gs, None, n_splits=N_FOLDS)

def test_alle(m, te_c, te_i, te_s):
    r = {"knie": kennzahlen(vorhersage(m, Xc[te_c]), yc[te_c]),
         "imb":  kennzahlen(vorhersage(m, Xi[te_i]), yi[te_i]),
         "si":   kennzahlen(vorhersage(m, Xs[te_s]), ys[te_s])}
    for pos, (X, y, _, _) in PIL.items():
        r[f"pilot_{pos}"] = kennzahlen(vorhersage(m, X), y)
    return r

VAR = ["ohne Fussdaten (B1r)", "+ ImbalanSense (B2r)", "+ ImbalanSense + Smart-Insole"]
siE = {k: [] for k in VAR}
for i, ((tr_c, te_c), (tr_i, te_i), (tr_s, te_s)) in enumerate(zip(sp_c, sp_i, sp_s), 1):
    t = time.time()
    # 1) nur Bein (CAPPIMU-Schienbein), augmentiert
    i_c = kappe(tr_c, yc, MAX_ADL_CAP)
    Xa, ya = fl.augment_orientation(Xc[i_c], yc[i_c], copies=1, seed=100 + i)
    m = fl.finetune(m_wk, fl.znorm(Xa), ya, freeze=0); del Xa
    siE[VAR[0]].append(test_alle(m, te_c, te_i, te_s)); del m
    # 2) Bein + ImbalanSense (= B2r)
    X, y = trainingssatz(tr_c, tr_i, aug=True)
    m = fl.finetune(m_wk, X, y, freeze=0)
    siE[VAR[1]].append(test_alle(m, te_c, te_i, te_s)); del m
    # 3) zusaetzlich Smart-Insole (Trainingsprobanden der Falte)
    i_s = rng.choice(tr_s, min(MAX_ADL_SI, len(tr_s)), replace=False)
    Xa, ya = fl.augment_orientation(Xs[i_s], ys[i_s], copies=1, seed=200 + i)
    X2 = np.concatenate([X, fl.znorm(Xa)]); y2 = np.concatenate([y, ya]); del X, Xa
    p = rng.permutation(len(y2))                  # validation_split nimmt das Ende
    m = fl.finetune(m_wk, X2[p], y2[p], freeze=0); del X2
    siE[VAR[2]].append(test_alle(m, te_c, te_i, te_s)); del m
    print(f"Falte {i}/{N_FOLDS} ({time.time()-t:.0f}s)")
    for k in VAR:
        r = siE[k][-1]
        print(f"  {k:<32} Knie Se {r['knie']['Se']:.3f}  IMB FP/h {r['imb']['fp_per_hour']:>6}"
              f"  SI FP/h {r['si']['fp_per_hour']:>6}")

## Ergebnis Teil E (Sensitivitaet, Genauigkeit, Fehlalarme)

In [ ]:
def pool_acc(liste, schl):
    tp = sum(r[schl]["TP"] for r in liste); fn = sum(r[schl]["FN"] for r in liste)
    fp = sum(r[schl]["FP"] for r in liste); tn = sum(r[schl]["TN"] for r in liste)
    h = (fp + tn) * fl.WIN_SEC * (1 - fl.OVERLAP) / 3600
    return {"Se": tp / (tp + fn) if tp + fn else None,
            "Se_ci95": fl.wilson_ci(tp, tp + fn) if tp + fn else None,
            "Sp": tn / (tn + fp) if tn + fp else None,
            "Acc": (tp + tn) / (tp + tn + fp + fn),
            "fp_per_hour": fp / h if h else None}

tab = {}
print(f"{'Variante':<32}{'Knie Se':>9}{'Knie Acc':>10}{'IMB Sp':>9}{'IMB FP/h':>10}{'SI Sp':>8}{'SI FP/h':>9}")
for k in VAR:
    z = {s: pool_acc(siE[k], s) for s in ["knie", "imb", "si"]}
    for pos in PIL:
        z[pos] = pool_acc(siE[k], f"pilot_{pos}")
    tab[k] = z
    print(f"{k:<32}{z['knie']['Se']:>9.3f}{z['knie']['Acc']:>10.3f}{z['imb']['Sp']:>9.4f}"
          f"{z['imb']['fp_per_hour']:>10.1f}{z['si']['Sp']:>8.4f}{z['si']['fp_per_hour']:>9.1f}")

vglE = {}
for a, b, s, met in [(VAR[0], VAR[1], "knie", "Se"), (VAR[0], VAR[1], "imb", "Sp"),
                     (VAR[0], VAR[1], "si", "Sp"),
                     (VAR[1], VAR[2], "knie", "Se"), (VAR[1], VAR[2], "imb", "Sp"),
                     (VAR[1], VAR[2], "si", "Sp")]:
    print(f"\n{b} vs {a} — {s}")
    vglE[f"{b}|{a}|{s}|{met}"] = fl.paired_comparison(
        [r[s] for r in siE[a]], [r[s] for r in siE[b]], a, b, metric=met)

fl.save_results(run_si, {"tabelle": tab, "vergleiche": vglE, "falten": siE},
    config={**fl.default_config(), "n_folds": N_FOLDS, "max_adl_si": MAX_ADL_SI,
            "augmentierung": "1 Kopie, Drehung + 50 % Spiegelung", "freeze": 0},
    notes=("Vorher/Nachher Fussdaten: ohne Fussdaten, +ImbalanSense, "
           "+ImbalanSense+Smart-Insole. SI-Test immer auf nicht trainierten "
           "SI-Probanden. Nur fuer den Bericht (SI-Lizenz nicht-kommerziell)."))